# 01 · Exploring FPL player data

**Question:** what drives a player's FPL points in a match, and how predictable are they?

Data: the public [Fantasy-Premier-League](https://github.com/vaastav/Fantasy-Premier-League) dataset, one row per player per fixture, seasons 2022-23 to 2026-27 (the seasons with expected-goals data).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fpl_ml import data, features, plots
plots.use_style()
FIG = Path("../reports/figures")

In [ ]:
df = data.load_history()
played = df[df.minutes > 0]
print(f"{len(df):,} player-fixture rows, {df.groupby(['season', 'element']).ngroups:,} player-seasons")
df.groupby("season").agg(rows=("element", "size"), gameweeks=("gw", "max"), players=("element", "nunique"))

## 1. Points are mostly zero, with a long tail
Most rows are players who did not play. Among those who did, 1-2 points (just turning up) is most common; hauls of 10+ are rare. This makes the target **noisy and skewed**: any model will be a few points off on most individual matches.

In [ ]:
share_zero_minutes = (df.minutes == 0).mean()
fig, ax = plt.subplots()
counts = played.total_points.clip(-3, 20).value_counts().sort_index()
ax.bar(counts.index, counts.values / counts.sum() * 100, width=0.8, color=plots.SERIES[0])
ax.set(title="Points per match, players who played", xlabel="FPL points (20 = 20 or more)", ylabel="% of matches")
fig.savefig(FIG / "points_distribution.png", bbox_inches="tight")
print(f"{share_zero_minutes:.0%} of rows are players who did not play at all.")
print(f"Among those who played: median {played.total_points.median():.0f} pts, {(played.total_points >= 10).mean():.1%} score 10+.")

## 2. Playing time is the biggest factor
60+ minutes earns the full appearance points and makes clean-sheet points possible, so expected minutes matter more than anything else.

In [ ]:
bands = pd.cut(df.minutes, [-1, 0, 59, 120], labels=["0 (didn't play)", "1-59", "60+"])
by_band = df.groupby(bands, observed=True).total_points.mean()
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(by_band.index.astype(str), by_band.values, color=plots.SERIES[0], width=0.6)
plots.label_bars(ax)
ax.set(title="Average points by minutes played", xlabel="Minutes in the match", ylabel="Average points")
fig.savefig(FIG / "points_by_minutes.png", bbox_inches="tight")

## 3. Position matters, and so does home advantage
Average points per match for players who played 60+ minutes.

In [ ]:
full = df[df.minutes >= 60]
tab = full.pivot_table(index="position", columns="was_home", values="total_points", aggfunc="mean").loc[data.POSITIONS]
tab.columns = ["Away", "Home"]
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(tab))
ax.bar(x - 0.2, tab["Home"], width=0.38, label="Home", color=plots.SERIES[0])
ax.bar(x + 0.2, tab["Away"], width=0.38, label="Away", color=plots.SERIES[1])
ax.set_xticks(x, tab.index)
plots.label_bars(ax)
ax.legend(loc="upper left", ncols=2)
ax.set(title="Average points per 60+ minute match", ylabel="Average points")
fig.savefig(FIG / "points_by_position_home_away.png", bbox_inches="tight")
tab.round(2)

## 4. Expected goals (xG) track real goals
Over a season, a player's total xG is a good guide to his goals. Single matches are noisy, but the underlying rate is real, which is why xG per 90 minutes is a model feature.

In [ ]:
season_totals = df.groupby(["season", "element"]).agg(xg=("xg", "sum"), goals=("goals_scored", "sum"))
season_totals = season_totals[season_totals.xg > 0.5]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(season_totals.xg, season_totals.goals, s=10, alpha=0.35, color=plots.SERIES[0], linewidths=0)
lim = [0, season_totals[["xg", "goals"]].max().max() + 1]
ax.plot(lim, lim, color=plots.TEXT_SECONDARY, linewidth=1, linestyle="--")
ax.annotate("goals = xG", (lim[1] * 0.72, lim[1] * 0.8), color=plots.TEXT_SECONDARY, fontsize=9)
ax.grid(axis="both")
ax.set(title="Season goals vs. season xG (one dot per player-season)", xlabel="Expected goals (xG)", ylabel="Goals")
fig.savefig(FIG / "xg_vs_goals.png", bbox_inches="tight")
print(f"correlation: {season_totals.xg.corr(season_totals.goals):.2f}")

## 5. Recent form predicts the next match, but only partly
Average next-match points, grouped by the player's average over his previous 5 matches. Form carries real signal (the line rises), yet the spread is large: the same form can give anything from 0 to 15 points.

In [ ]:
f = features.build_features(df)
f = f[f.total_points_last5.notna()]
bucket = pd.cut(f.total_points_last5, [-3, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 7, 20])
trend = f.groupby(bucket, observed=True).total_points.agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(7.5, 4))
mids = [b.mid for b in trend.index]
ax.plot(mids, trend["mean"], marker="o", markersize=8, color=plots.SERIES[0])
ax.set(title="Next-match points vs. last-5 average", xlabel="Average points over previous 5 matches", ylabel="Average next-match points")
fig.savefig(FIG / "form_persistence.png", bbox_inches="tight")
print(f"correlation of last-5 average with next-match points: {f.total_points_last5.corr(f.total_points):.2f}")
trend.round(2)

## Takeaways for modelling
- **Minutes first:** whether and how long a player plays explains most of the points; recent minutes and starts are key features.
- **Underlying numbers:** xG and xA per 90 describe attacking threat better than goals and assists.
- **Context:** position, home/away and the strength of both teams shift the expectation.
- **Noise:** even perfect knowledge of these leaves large match-to-match variation, so the realistic goal is to be *less wrong on average* and to *rank players well*, not to predict exact scores.